# Notebook for reducing the dimensionality of data.

Run interactive notebooks in the following order:\
preprocessing.ipynb  ->  plot_corr_assoc.ipynb  ->  dimensionality_reduction.ipynb  ->  clustering.ipynb

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from utils.utils import check_save_path, load_data, save_data
from utils.dimensionality_reduction_utils import plot_reduced_data, reduce_dimen

pd.options.mode.copy_on_write = True

In [ ]:
# Create RNG for reproducibility
rng = np.random.default_rng(3295)

# Hardcoded paths, change to apply to new data
data_path = Path("data/preprocessed/esa_2018_2021_imputed.pkl")
vars_info_path = Path("data/preprocessed/esa_vars_info.pkl")

image_folder = "outputs/dim_red"
dim_red_plot_save_name_template = "dim_red_{group}.png"
scree_plot_save_name_template = "scree_{group}.png"

outputs_folder = "outputs/dim_red"
model_save_name_template = "esa_2018_2021_model_{group}.pkl"
data_reduced_save_name_template = "esa_2018_2021_reduced_{group}.pkl"
merged_data_reduced_save_path = Path(outputs_folder, "esa_2018_2021_reduced.pkl")

allow_overwrite = True

# Read in data
data: pd.DataFrame = load_data(data_path)
vars_info: pd.DataFrame = load_data(vars_info_path)

In [ ]:
dim_red_methods = {
    "socioecon": "FAMD", "alcohol": "PCA", "nicotine": "PCA", "other_drugs": "PCA", "medication": "PCA"
}  # fmt: skip

dim_reduced_data: dict[str, pd.DataFrame] = {}

for group, dim_red_method in dim_red_methods.items():
    # Restrict to one group
    group_vars = vars_info.loc[vars_info["group"] == group, "name"]
    group_var_types = vars_info.loc[vars_info["group"] == group, "type"]
    group_data = data[group_vars]

    # Conduct dimensionality reduction
    treat_cat_as_continuous: list[str] = vars_info.loc[
        (vars_info["group"] == group) & vars_info["type"].isin(["binary", "ordinal"]), "name"
    ].tolist()
    data_loadings, model, var_loadings = reduce_dimen(
        group_data, dim_red_method, n_components=4, treat_cat_as_continuous=treat_cat_as_continuous, rng=rng
    )

    # Visualize data and variables in coordinates of the first two components
    coordinates_plot = plot_reduced_data(
        data_loadings,
        var_loadings,
        comp_x_id=0,
        comp_y_id=1,
        var_marker="label",
        # var_marker="x",
    )
    # import altair as alt
    # alt.data_transformers.disable_max_rows()
    # coordinates_plot = model.plot(group_data, x_component=0, y_component=1, show_column_labels=True)
    coordinates_plot.show()
    dim_red_plot_save_path = Path(image_folder, dim_red_plot_save_name_template.format(group=group))
    check_save_path(dim_red_plot_save_path, allow_overwrite=allow_overwrite)
    coordinates_plot.save(dim_red_plot_save_path)
    print(f"Saved plot of reduced {group} {dim_red_method} data as PNG to {dim_red_plot_save_path}.")

    # Investigate eigenvalues and show scree plot
    print(model.eigenvalues_summary)
    scree_plot = model.scree_plot()
    scree_plot.show()
    scree_plot_save_path = Path(image_folder, scree_plot_save_name_template.format(group=group))
    check_save_path(scree_plot_save_path, allow_overwrite=allow_overwrite)
    scree_plot.save(scree_plot_save_path)
    print(f"Saved {group} {dim_red_method} Scree plot as PNG to {scree_plot_save_path}.")

    # Save fitted model and data with reduced dimensionality (factor loadings of data)
    model_save_path = Path(outputs_folder, model_save_name_template.format(group=group))
    save_data(model, model_save_path, label=f"fitted {group} {dim_red_method} model")

    data_reduced_save_path = Path(outputs_folder, data_reduced_save_name_template.format(group=group))
    save_data(
        data_loadings,
        data_reduced_save_path,
        label=f"{group} data with reduced dimensionality ({dim_red_method} factor loadings of data)",
    )

    dim_reduced_data[group] = data_loadings.rename(columns=lambda x: f"{group}_comp_{x}")

# Merge data from groups
merged_data_reduced = pd.concat(dim_reduced_data.values(), axis=1)
# import itertools
# non_reduced_vars = vars_info.loc[~ vars_info["group"].isin(dim_reduced_data.keys()), "name"]
# merged_data_reduced = pd.concat(itertools.chain((data[non_reduced_vars],), dim_reduced_data.values()), axis=1)
save_data(
    merged_data_reduced,
    merged_data_reduced_save_path,
    label="merged data from all groups with reduced dimensionality",
)